In [64]:
from jaad_data import JAAD

jaad_api = JAAD(data_path = '.')

In [62]:
# Extract & Save Images
jaad_api.extract_and_save_images()

video_0001
[####################] 99.83% 

video_0002
[####################] 99.52% 

video_0003
[####################] 99.52% 

video_0004
[####################] 99.44% 

video_0005
[####################] 99.58% 

video_0006
[####################] 99.70% 

video_0007
[####################] 99.17% 

video_0008
[####################] 99.33% 

video_0009
[####################] 99.17% 

video_0010
[####################] 98.89% 

video_0011
[####################] 99.63% 

video_0012
[####################] 99.17% 

video_0013
[####################] 99.33% 

video_0014
[####################] 99.63% 

video_0015
[####################] 99.76% 

video_0016
[####################] 99.52% 

video_0017
[####################] 99.63% 

video_0018
[####################] 99.74% 

video_0019
[####################] 99.79% 

video_0020
[####################] 99.81% 

video_0021
[####################] 99.44% 

video_0022
[####################] 99.76% 

video_0023
[####################] 99.58% 

video_0024


KeyboardInterrupt: 

In [2]:
# Generate Database
db = jaad_api.generate_database()

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\ASUS\Documents\Year 3 Semester 2\JAAD\data_cache\jaad_database.pkl


In [30]:
# Extract Features from Pedestrians
# Notes on meaning of Behavior Annotations:
    # occlusion: 0(not occluded), 1(partially occluded), 2(fully occluded)
    # cross: 0(not crossing), 1(crossing)
    # reaction: 0(no reaction), 1(reaction)
    # hand_gesture: 0(no hand gesture), 1(hand gesture)
    # look: 0(not looking), 1(looking)
    # action: 0(Standing), 1(Walking)
    # nod: 0(no nod), 1(nod)
# Notes on meaning of Pedestrian Attributes:
    # old_id: Original Annotation ID String
    # age: 0(child), 1 (young), 2 (adult), 3 (senior)
    # crossing: 0(not crossing), 1(crossing), -1(irrelevant)
    # crossing_point: Frame Index of Crossing Point (if crossing), -1 otherwise
    # decision_point: Frame Index of Decision Point (if crossing), -1 otherwise
    # designated: 0(not designated crossing point), 1(designated crossing point)
    # gender: 0(n/a), 1(female), 2(male)
    # group_size: Number of People in Group
    # intersection: 0(not at intersection), 1(at intersection)
    # motion_direction: 0(n/a), 1(Lateral / Across), 2(Longitudinal / Along)
    # num_lanes: Number of Road Lanes
    # signalized: 0(n/a), 1(non-signalized intersection), 2(signalized intersection)
    # traffic_direction: 0(One-Way), 1(Two-Way)

pedestrian_ids = jaad_api._get_pedestrian_ids()

features = []
for vid, video in db.items():
    for pid, pedestrian in video["ped_annotations"].items():
        if 'b' not in pid: # Skip Pedestrians without Behavior Annotations
            continue
        
        # Dynamic Features (Time-Series)
        frames = pedestrian.get("frames", [])
        bboxes = pedestrian.get("bbox", [])
        occlusions = pedestrian.get("occlusion", [])
        behavior = pedestrian.get("behavior", {})

        # Static Features (Non-Time-Series)
        attributes = pedestrian.get("attributes", {})


        for i, frame in enumerate(frames):
            row = {
                "video_id": vid,
                "pedestrian_id": pid,

                # Frames
                "frame_id": frame,

                # BBoxes
                "bbox_x1": bboxes[i][0],
                "bbox_y1": bboxes[i][1],
                "bbox_x2": bboxes[i][2],
                "bbox_y2": bboxes[i][3],

                # Occlusions
                "occlusion": occlusions[i],

                # Behavior Annotations
                "cross": behavior.get("cross", [0]*len(frames))[i],
                "reaction": behavior.get("reaction", [0]*len(frames))[i],
                "hand_gesture": behavior.get("hand_gesture", [0]*len(frames))[i],
                "look": behavior.get("look", [0]*len(frames))[i],
                "action": behavior.get("action", [0]*len(frames))[i],
                "nod": behavior.get("nod", [0]*len(frames))[i],

                # Pedestrian Attributes
                "old_id": attributes.get("old_id", ""),
                "age": attributes.get("age", 0),
                "crossing": attributes.get("crossing", 0),
                "crossing_point": attributes.get("crossing_point", 0),
                "decision_point": attributes.get("decision_point", 0),
                "designated": attributes.get("designated", 0),
                "gender": attributes.get("gender", 0),
                "group_size": attributes.get("group_size", 1),
                "intersection": attributes.get("intersection", 0),
                "motion_direction": attributes.get("motion_direction", 0),
                "num_lanes": attributes.get("num_lanes", 0),
                "signalized": attributes.get("signalized", 0),
                "traffic_direction": attributes.get("traffic_direction", 0)
            }
            
            features.append(row)


import pandas as pd

df = pd.DataFrame(features)
print(f"Features shape: {df.shape}")
display(df.head())

# Count number of unique pedestrians
unique_pedestrians = df["pedestrian_id"].nunique()
assert unique_pedestrians == len([pid for pid in pedestrian_ids if 'b' in pid]), "Mismatch between unique pedestrians in features and pedestrian IDs."

# Check for Class Imbalance in Crossing Behavior
crossing_counts = df["crossing"].value_counts()
display(crossing_counts)

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\ASUS\Documents\Year 3 Semester 2\JAAD\data_cache\jaad_database.pkl
Features shape: (132700, 27)


,video_id,pedestrian_id,frame_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,occlusion,cross,reaction,...,crossing_point,decision_point,designated,gender,group_size,intersection,motion_direction,num_lanes,signalized,traffic_direction
0,video_0001,0_1_3b,0,465.0,730.0,533.0,848.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
1,video_0001,0_1_3b,1,463.0,730.0,532.0,848.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
2,video_0001,0_1_3b,2,461.0,730.0,531.0,849.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
3,video_0001,0_1_3b,3,459.0,730.0,530.0,849.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
4,video_0001,0_1_3b,4,458.0,731.0,530.0,851.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1


crossing
 1    105026
-1     16216
 0     11458
Name: count, dtype: int64

In [59]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# Feature engineering + training (frame-level target: cross)
df = pd.DataFrame(features).copy()
df = df.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)

# JAAD frames are 1920x1080
W, H = 1920.0, 1080.0

# Normalized bbox geometry
df["bbox_center_x"] = ((df["bbox_x1"] + df["bbox_x2"]) / 2.0) / W
df["bbox_center_y"] = ((df["bbox_y1"] + df["bbox_y2"]) / 2.0) / H
df["bbox_width"] = (df["bbox_x2"] - df["bbox_x1"]) / W
df["bbox_height"] = (df["bbox_y2"] - df["bbox_y1"]) / H
df["bbox_area"] = df["bbox_width"] * df["bbox_height"]
df["aspect_ratio"] = df["bbox_width"] / (df["bbox_height"] + 1e-6)

group_keys = ["video_id", "pedestrian_id"]

# Velocity from normalized centers
df["velocity_x"] = df.groupby(group_keys)["bbox_center_x"].diff().fillna(0.0)
df["velocity_y"] = df.groupby(group_keys)["bbox_center_y"].diff().fillna(0.0)
df["speed"] = np.sqrt(df["velocity_x"] ** 2 + df["velocity_y"] ** 2)

# Acceleration from normalized velocity
df["acceleration_x"] = df.groupby(group_keys)["velocity_x"].diff().fillna(0.0)
df["acceleration_y"] = df.groupby(group_keys)["velocity_y"].diff().fillna(0.0)

# Frame-level label
df = df[df["cross"].isin([0, 1])].copy()
y = df["cross"]

# Keep names consistent for training and inference
cat_cols = [
    "age", "designated", "gender", "intersection",
    "motion_direction", "signalized", "traffic_direction"
]
num_cols = [
    "bbox_center_x", "bbox_center_y", "bbox_width", "bbox_height",
    "bbox_area", "aspect_ratio",
    "velocity_x", "velocity_y", "speed", "acceleration_x", "acceleration_y",
    "occlusion", "reaction", "hand_gesture", "look", "action", "nod",
    "group_size", "num_lanes"
]

X = df[cat_cols + num_cols]
groups = df["video_id"]

# Group split to reduce scene leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

clf = Pipeline([
    ("pre", pre),
    ("model", XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.025,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
    ))
])

clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("AUROC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))

AUROC: 0.9504912142850761
              precision    recall  f1-score   support

           0       0.90      0.85      0.87     11116
           1       0.89      0.92      0.91     14379

    accuracy                           0.89     25495
   macro avg       0.89      0.89      0.89     25495
weighted avg       0.89      0.89      0.89     25495



In [ ]:
# Visualization of Predictions on Video Frames
# This Code was generated by AI
# Temporary Placeholder for Video Visualization Code - To be implemented

import numpy as np
import pandas as pd
import cv2
import os

def predict_from_df(df, clf, cat_cols, num_cols, video_id=None, pedestrian_id=None):
    """Return per-frame crossing probabilities directly from the feature DataFrame."""
    data = df.copy()

    if video_id is not None:
        data = data[data["video_id"] == video_id].copy()
    if pedestrian_id is not None:
        data = data[data["pedestrian_id"] == pedestrian_id].copy()

    if data.empty:
        raise ValueError("No rows matched the requested video_id / pedestrian_id filter.")

    data = data.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)

    # Recompute the same features used during training if they are not already present
    if "bbox_center_x" not in data.columns:
        W, H = 1920.0, 1080.0
        group_keys = ["video_id", "pedestrian_id"]
        data["bbox_center_x"] = ((data["bbox_x1"] + data["bbox_x2"]) / 2.0) / W
        data["bbox_center_y"] = ((data["bbox_y1"] + data["bbox_y2"]) / 2.0) / H
        data["bbox_width"] = (data["bbox_x2"] - data["bbox_x1"]) / W
        data["bbox_height"] = (data["bbox_y2"] - data["bbox_y1"]) / H
        data["bbox_area"] = data["bbox_width"] * data["bbox_height"]
        data["aspect_ratio"] = data["bbox_width"] / (data["bbox_height"] + 1e-6)
        data["velocity_x"] = data.groupby(group_keys)["bbox_center_x"].diff().fillna(0.0)
        data["velocity_y"] = data.groupby(group_keys)["bbox_center_y"].diff().fillna(0.0)
        data["speed"] = np.sqrt(data["velocity_x"] ** 2 + data["velocity_y"] ** 2)
        data["acceleration_x"] = data.groupby(group_keys)["velocity_x"].diff().fillna(0.0)
        data["acceleration_y"] = data.groupby(group_keys)["velocity_y"].diff().fillna(0.0)

    X = data.reindex(columns=cat_cols + num_cols, fill_value=0)
    scores = clf.predict_proba(X)[:, 1]
    result = data[[
        "video_id", "pedestrian_id", "frame_id",
        "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
        "cross"
    ]].copy()
    result["crossing_score"] = scores
    result["predicted_cross"] = (scores >= 0.5).astype(int)
    return result

def write_prediction_video_from_df(df, obj, clf, cat_cols, num_cols, video_id, output_path, fps=30):
    """Write a video with bounding boxes, model prediction, and ground truth per frame."""
    predictions = predict_from_df(df, clf, cat_cols, num_cols, video_id=video_id)

    images_root = getattr(obj, "_images_path", os.path.join(".", "images"))
    video_path = os.path.join(images_root, video_id)
    if not os.path.isdir(video_path):
        raise FileNotFoundError(f"Video image folder not found: {video_path}")

    frame_files = sorted([
        f for f in os.listdir(video_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    if not frame_files:
        raise RuntimeError(f"No frames found in {video_path}. Expected .jpg or .png files.")

    first_frame = cv2.imread(os.path.join(video_path, frame_files[0]))
    if first_frame is None:
        raise RuntimeError(f"Could not read first frame: {frame_files[0]}")

    height, width = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer for: {output_path}")

    pred_by_frame = {}
    for _, row in predictions.iterrows():
        pred_by_frame.setdefault(int(row["frame_id"]), []).append(row)

    for frame_file in frame_files:
        frame_idx = int(os.path.splitext(frame_file)[0])
        img = cv2.imread(os.path.join(video_path, frame_file))
        if img is None:
            continue

        for row in pred_by_frame.get(frame_idx, []):
            bbox = [row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]]
            score = float(row["crossing_score"])
            pred_label = int(row["predicted_cross"])
            gt_label = row["cross"]
            color = (0, int(255 * (1 - score)), int(255 * score))
            cv2.rectangle(img, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), color, 3)
            label = f"{row['pedestrian_id']} Pred: {pred_label} GT: {gt_label} Score: {score:.2f}"
            cv2.putText(
                img, label, (int(bbox[0]), max(20, int(bbox[1] - 10))),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
            )

        writer.write(img)

    writer.release()
    return output_path

# Example: generate frame-level predictions and write a video
df_predictions = predict_from_df(df, clf, cat_cols, num_cols, video_id="video_0001")
# display(df_predictions.head(20))

output_video_path = os.path.join(".", "video_0001_predictions.mp4")
written_path = write_prediction_video_from_df(
    df, jaad_api, clf, cat_cols, num_cols, video_id="video_0001", output_path=output_video_path
 )
print(f"Saved prediction video to: {written_path}")

ValueError: No rows matched the requested video_id / pedestrian_id filter.